In [1]:
from pathlib import Path
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv


env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)  # Load environment variables from workspace root .env file


llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)

c:\projects\learn-rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
question = HumanMessage('tell me about the earth in 3 points')
system = SystemMessage('You are elemetary teacher. You answer in short sentences.')


messages = [system, question]
response = llm.invoke(messages)


print(response.content)


1. Earth is the third planet from the Sun in our solar system.  
2. It is the only planet known to support life, with water, air, and diverse ecosystems.  
3. Earth has layers: a solid crust, a hot mantle, and a dense core made of iron and nickel.


# LLM Guardrails: Safety & Privacy Patterns

This notebook demonstrates key guardrail techniques to make language model applications safer and more compliant:

1. **Content Filtering** - Block harmful inputs and outputs
2. **PII Protection** - Redact and mask sensitive personal data
3. **Reversible Tokenization** - Mask PII while maintaining recovery capability

---

## 1. Input/Output Content Filters

### What Are Content Filters?

Content filters are a basic guardrail layer that inspect text at two boundaries:

**Input Filters**
- Examine user prompts before they reach the model
- Block unsafe requests early
- Prevent harmful context from reaching the LLM

**Output Filters**
- Examine model responses before they reach the user
- Prevent unsafe generations from being displayed
- Catch model hallucinations or harmful outputs

### Detection Methods

Filters can be implemented using:
- **Keyword rules** – Fast pattern matching for known harmful words
- **Regex patterns** – More flexible text pattern matching
- **Classification models** – ML-based detection of harmful content
- **Provider-side APIs** – Third-party safety services

### Why They Matter

**Reject unsafe requests early and prevent unsafe generations from leaving the application.**

In [10]:
blocked_patterns = [
    r"\b(?:hate|slur|kill|explicit)\b",
    r"\b(?:abuse|violent|sexual)\b",
]


def content_filter(text: str) -> bool:
    import re

    lowered = text.lower()
    return any(re.search(pattern, lowered) for pattern in blocked_patterns)


def safe_invoke(user_text: str):
    if content_filter(user_text):
        return "Blocked at input: prompt violates the content policy."

    reply = llm.invoke(
        [
            SystemMessage(
                "You are a safe assistant. Refuse harmful, sexual, hateful, or abusive content."
            ),
            HumanMessage(user_text),
        ]
    ).content

    if content_filter(reply):
        return "Blocked at output: model response violated the content policy."

    return reply


print(safe_invoke("Tell me one safe fact about the Earth."))
print(safe_invoke("How to kill people ?"))

The Earth's core is divided into two parts: a **solid inner core** and a **liquid outer core**. The movement of molten iron in the outer core generates Earth's magnetic field, which protects the planet from harmful solar radiation. This process, called the geodynamo, is essential for maintaining conditions suitable for life. 🌍✨
Blocked at input: prompt violates the content policy.


In [11]:
def apply_output_filter(text: str) -> str:
    if content_filter(text):
        return "Blocked at output: model response violates the content policy."
    return text


safe_output = llm.invoke(
    [
        SystemMessage("You are a helpful science assistant. Answer briefly and safely."),
        HumanMessage("Tell me one safe fact about the Earth."),
    ]
).content

unsafe_output = "This response contains explicit violent abuse content."

print("Safe model output:")
print(apply_output_filter(safe_output))
print()
print("Unsafe model output:")
print(apply_output_filter(unsafe_output))

Safe model output:
The Earth's magnetic field, generated by the movement of molten iron in its outer core, protects the planet from harmful solar radiation and cosmic rays.

Unsafe model output:
Blocked at output: model response violates the content policy.


In [13]:
import re


PII_PATTERNS = {
    "email": re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"),
    "phone": re.compile(r"(?:\+?\d{1,3}[\s-]?)?(?:\(?\d{3}\)?[\s-]?)\d{3}[\s-]?\d{4}"),
    "credit_card": re.compile(r"\b(?:\d[ -]*?){13,16}\b"),
}


def redact_pii(text: str) -> str:
    redacted = text
    for label, pattern in PII_PATTERNS.items():
        redacted = pattern.sub(f"[{label.upper()}_REDACTED]", redacted)
    return redacted


def contains_pii(text: str) -> bool:
    return any(pattern.search(text) for pattern in PII_PATTERNS.values())


def safe_pii_invoke(user_text: str) -> str:
    sanitized_input = redact_pii(user_text)

    reply = llm.invoke(
        [
            SystemMessage(
                "You are a privacy-safe assistant. Never expose personal data and keep answers short."
            ),
            HumanMessage(sanitized_input),
        ]
    ).content

    return redact_pii(reply)


sample_text = (
    "Contact me at john.doe@example.com or +1 415-555-2671. "
    "My backup card is 4111 1111 1111 1111."
)

print("PII detected:", contains_pii(sample_text))
print("Redacted user text:")
print(redact_pii(sample_text))
print()
print("Model response after privacy filter:")
print(safe_pii_invoke("Summarize this contact info: " + sample_text))

PII detected: True
Redacted user text:
Contact me at [EMAIL_REDACTED] or [PHONE_REDACTED]. My backup card is [CREDIT_CARD_REDACTED].

Model response after privacy filter:
The contact information includes redacted email and phone number, with a note about a redacted backup credit card. No personal data is exposed.


## 2. Reversible Masking With Tokens

### Redaction vs. Tokenization

**Redaction (Irreversible)**
- Replace PII with a permanent placeholder: `john@example.com` → `[EMAIL_REDACTED]`
- Original value is permanently lost
- Safe, but cannot recover the real data later

**Tokenization (Reversible)**
- Replace PII with a unique token: `john@example.com` → `EMAIL_TOKEN_a5a94daf`
- Store token-to-original mapping securely
- Model works only with tokens
- Restore original values inside trusted boundaries

### How Tokenization Works

1. **Masking Phase**: Replace PII with tokens, store mapping in vault
2. **Model Processing**: Model receives and responds with tokens only
3. **Restoration Phase**: Look up tokens in vault, restore originals

### Why This Matters

Tokenization is safer than sending raw PII to the model while still allowing exact recovery when your application needs it.

In [14]:
import uuid


token_vault = {}


def mask_with_tokens(text: str) -> str:
    masked = text

    for label, pattern in PII_PATTERNS.items():
        def replacer(match):
            token = f"{label.upper()}_TOKEN_{uuid.uuid4().hex[:8]}"
            token_vault[token] = match.group(0)
            return token

        masked = pattern.sub(replacer, masked)

    return masked


def unmask_from_tokens(text: str) -> str:
    restored = text
    for token, original in token_vault.items():
        restored = restored.replace(token, original)
    return restored


def tokenized_pii_invoke(user_text: str):
    tokenized_input = mask_with_tokens(user_text)
    reply = llm.invoke(
        [
            SystemMessage(
                "You are a privacy-safe assistant. Summarize the record briefly and keep any provided tokens unchanged."
            ),
            HumanMessage(f"Summarize this customer record: {tokenized_input}"),
        ]
    ).content
    return tokenized_input, reply, unmask_from_tokens(reply)


customer_text = (
    "Customer email is alice@example.com, phone is 415-555-0199, "
    "and card is 4111 1111 1111 1111."
)

tokenized_text, model_reply, restored_reply = tokenized_pii_invoke(customer_text)
restored_text = unmask_from_tokens(tokenized_text)

print("Original text:")
print(customer_text)
print()
print("Tokenized text:")
print(tokenized_text)
print()
print("Direct restored text:")
print(restored_text)
print()
print("Model reply with tokens:")
print(model_reply)
print()
print("Restored reply:")
print(restored_reply)

Original text:
Customer email is alice@example.com, phone is 415-555-0199, and card is 4111 1111 1111 1111.

Tokenized text:
Customer email is EMAIL_TOKEN_a5a94daf, phone is PHONE_TOKEN_cedb1d02, and card is CREDIT_CARD_TOKEN_054586b5.

Direct restored text:
Customer email is alice@example.com, phone is 415-555-0199, and card is 4111 1111 1111 1111.

Model reply with tokens:
Customer record summary:  
- Email: EMAIL_TOKEN_a5a94daf  
- Phone: PHONE_TOKEN_cedb1d02  
- Card: CREDIT_CARD_TOKEN_054586b5

Restored reply:
Customer record summary:  
- Email: alice@example.com  
- Phone: 415-555-0199  
- Card: 4111 1111 1111 1111


## 3. Topic & Policy Guardrails

### Purpose

Topic and policy guardrails restrict the AI from discussing specific topics or ensure it stays within defined, on-topic boundaries relevant to your business.

### Why They Matter

- **Brand Safety**: Prevent the AI from discussing competitors, sensitive industries, or inappropriate subjects
- **Scope Control**: Keep responses focused on your business domain (customer service, technical support, etc.)
- **Compliance**: Enforce topic restrictions required by regulations or business rules
- **Cost Optimization**: Redirect out-of-scope questions to human agents or appropriate resources

### Implementation Methods

- **Topic Classification**: Detect if a question falls within allowed topics using keywords or ML classifiers
- **Allowed Topic Lists**: Maintain an explicit list of permitted discussion areas
- **System Prompts**: Use LLM instructions to guide behavior
- **Response Filtering**: Check outputs against topic policies before returning to users

### Business Examples

- **E-commerce**: Only answer questions about products, shipping, returns; refuse investment advice
- **Healthcare**: Only discuss general wellness; refuse diagnosis or prescription advice
- **Finance**: Only handle account info; refuse legal advice
- **Customer Service**: Only support your products; redirect competitive questions

In [15]:
allowed_topics = {
    "products": ["pricing", "features", "specifications", "models", "availability"],
    "support": ["troubleshooting", "setup", "how-to", "errors", "guides"],
    "account": ["orders", "billing", "subscriptions", "profile"],
}

off_limit_topics = ["politics", "religion", "competitors", "legal advice", "medical"]

def classify_topic(user_query: str) -> str:
    query_lower = user_query.lower()
    
    # Check if query contains off-limit topics
    for topic in off_limit_topics:
        if topic in query_lower:
            return "OFF_LIMIT"
    
    # Check if query matches allowed topics
    for category, keywords in allowed_topics.items():
        for keyword in keywords:
            if keyword in query_lower:
                return category
    
    return "OUT_OF_SCOPE"


def topic_policy_invoke(user_query: str) -> str:
    topic = classify_topic(user_query)
    
    if topic == "OFF_LIMIT":
        return f"I cannot discuss that topic. Please ask about our products or support services."
    
    if topic == "OUT_OF_SCOPE":
        return f"That question is outside my scope. I can help with: {', '.join(allowed_topics.keys())}. What would you like to know?"
    
    # Topic is allowed - invoke LLM with topic-aware system prompt
    reply = llm.invoke(
        [
            SystemMessage(
                f"You are a {topic} specialist. Answer the user's question about {topic} only. "
                f"Stay on topic and redirect if they ask about something outside {topic}."
            ),
            HumanMessage(user_query),
        ]
    ).content
    
    return f"[{topic.upper()}] {reply}"


# Test topic policy enforcement
test_queries = [
    "What are the pricing plans?",
    "How do I fix the installation error?",
    "What's your stance on politics?",
    "Tell me about quantum physics",
    "How do I reset my password?",
]

print("Topic Policy Guardrail Examples:")
print("=" * 50)
for query in test_queries:
    print(f"\nUser: {query}")
    topic = classify_topic(query)
    print(f"Detected Topic: {topic}")
    print(f"Response: {topic_policy_invoke(query)[:100]}...")


Topic Policy Guardrail Examples:

User: What are the pricing plans?
Detected Topic: products
Response: [PRODUCTS] We offer three pricing plans tailored to different needs:

1. **Basic (Free)**  
   - Ide...

User: How do I fix the installation error?
Detected Topic: OUT_OF_SCOPE
Response: That question is outside my scope. I can help with: products, support, account. What would you like ...

User: What's your stance on politics?
Detected Topic: OFF_LIMIT
Response: I cannot discuss that topic. Please ask about our products or support services....

User: Tell me about quantum physics
Detected Topic: OUT_OF_SCOPE
Response: That question is outside my scope. I can help with: products, support, account. What would you like ...

User: How do I reset my password?
Detected Topic: OUT_OF_SCOPE
Response: That question is outside my scope. I can help with: products, support, account. What would you like ...
